# Part 10 (실습) — 메모리: 기억하는 에이전트

> **이 노트북의 목표**
> Part 9 상담 에이전트에 `InMemorySaver`를 붙여 "아까 그 상품"이 통하게 만듦. **같은 thread_id면 맥락 유지, 다른 thread_id면 분리**되는 것을 직접 비교함.
>
> **사용 모델**: `gemini-3-flash` · **선행**: Part 0~9 (중급의 마지막 개념)

## 0. 준비

In [ ]:
# ─── 패키지 설치 ───
# %pip install: 주피터 노트북 안에서 파이썬 패키지를 설치하는 매직 명령어
#   -q (quiet): 설치 로그 최소화 | -U (upgrade): 최신 버전으로 업그레이드
%pip install -q -U langchain langchain-google-genai

In [ ]:
# os: 파이썬 내장 모듈 — 환경 변수(GOOGLE_API_KEY 등)를 읽고 쓸 때 사용
import os
# getpass: 입력한 글자를 화면에 숨겨서 받는 함수 (비밀번호 입력처럼)
#   → API 키를 노출 없이 안전하게 입력받기 위해 사용
from getpass import getpass
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Gemini API 키: ")
# ChatGoogleGenerativeAI: Google Gemini 모델을 LangChain에서 사용할 수 있게 감싸주는 클래스
#   → langchain-google-genai 패키지에서 제공
#   → .invoke() / .stream() / .batch() 등 LangChain 공통 메서드를 지원
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
# @tool 데코레이터: 일반 파이썬 함수를 LangChain "도구"로 변환
#   → 함수의 docstring이 도구 설명이 되고, 파라미터가 도구 입력 스키마가 됨
#   → 에이전트가 이 설명을 읽고 언제 이 도구를 쓸지 스스로 판단함
from langchain_core.tools import tool
# ─── Gemini 모델 인스턴스 생성 ───
# model: 사용할 Gemini 모델명 (예: "gemini-3-flash" — 빠르고 저렴한 기본 모델)
# temperature: 답변의 무작위성 조절 (0=일관된 답 / 1=창의적·다양한 답)
model = ChatGoogleGenerativeAI(model="gemini-3-flash", temperature=0)

@tool
def get_stock(product: str) -> str:
    """상품명으로 재고를 조회한다. 재고가 있는지 물을 때 사용한다."""
    db = {"베이지 니트": 12, "트렌치 코트": 0}
    return f"{product} 재고: {db.get(product, '정보 없음')}"

print("준비 완료")

## 1. 기억 없는 에이전트 (Part 9의 한계)

체크포인터 없이 만든 에이전트는 두 번째 호출이 첫 번째를 모름.

In [ ]:
agent_no_mem = create_agent(model=model, tools=[get_stock],
                            system_prompt="너는 리리마켓 상담원이야. 간결하게 답해.")

# .invoke(): 입력을 모델/체인에 보내고 결과를 받는 LangChain의 핵심 실행 메서드
#   → 모든 LangChain 부품(Runnable)이 이 메서드를 공유함
r1 = agent_no_mem.invoke({"messages": [("user", "베이지 니트 재고 있어요?")]})
print("1턴 답:", r1["messages"][-1].content)

r2 = agent_no_mem.invoke({"messages": [("user", "그거 반품 되나요?")]})
print("2턴 답:", r2["messages"][-1].content)
print("→ '그거'가 뭔지 모름. 두 호출이 독립적이라 첫 대화를 기억 못 함.")

## 2. 체크포인터 장착 — InMemorySaver

`create_agent`의 `checkpointer` 인수에 `InMemorySaver()`를 넣음.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=[get_stock],
    system_prompt="너는 리리마켓 상담원이야. 간결하게 답해.",
    checkpointer=InMemorySaver(),   # ← 메모리 장착
)
print("메모리 장착 완료")

## 3. thread_id로 맥락 잇기

호출 시 `config`에 `thread_id`를 넘김. 같은 thread_id면 기억이 이어짐.

In [ ]:
cfg = {"configurable": {"thread_id": "session-1"}}

r1 = agent.invoke({"messages": [("user", "베이지 니트 재고 있어요?")]}, config=cfg)
print("1턴:", r1["messages"][-1].content)

# 같은 thread_id → "그거"를 기억함
r2 = agent.invoke({"messages": [("user", "그거 색상은 뭐뭐 있어요?")]}, config=cfg)
print("2턴:", r2["messages"][-1].content)
print("→ 같은 thread_id라 '그거'=베이지 니트로 이해함!")

## 4. thread_id가 다르면 분리된다

다른 thread_id로 물으면, 앞 대화를 모르는 새 대화가 됨. (사용자 분리의 원리)

In [ ]:
cfg2 = {"configurable": {"thread_id": "session-2"}}   # 다른 thread

# session-1에서 베이지 니트를 얘기했지만, session-2는 모름
r = agent.invoke({"messages": [("user", "그거 색상 뭐 있어요?")]}, config=cfg2)
print("[session-2] 2턴:", r["messages"][-1].content)
print("→ 다른 thread라 '그거'가 뭔지 모름. 대화가 완전히 분리됨.")

## 5. 한 에이전트로 여러 사용자를 섞이지 않게

thread_id별로 대화가 분리되므로, 같은 에이전트가 여러 손님을 동시에 응대해도 섞이지 않음.

In [ ]:
def chat(thread, msg):
    cfg = {"configurable": {"thread_id": thread}}
    r = agent.invoke({"messages": [("user", msg)]}, config=cfg)
    print(f"[{thread}] 🙋 {msg}\n         🤖 {r['messages'][-1].content}\n")

# 김 손님(A)과 이 손님(B)이 번갈아 대화 — 섞이지 않음
chat("손님A", "베이지 니트 재고 있어요?")
chat("손님B", "트렌치 코트 재고 있어요?")
chat("손님A", "그거 반품 기간 얼마예요?")   # A의 '그거' = 베이지 니트
chat("손님B", "그거 언제 입고돼요?")        # B의 '그거' = 트렌치 코트

> 📌 같은 에이전트 객체 하나로 손님A·손님B를 동시에 응대해도, thread_id가 다르니 각자의 '그거'를 정확히 구분함. 이것이 실제 서비스의 기본 구조임.

## 6. 상태(state) 들여다보기 — 고급 예고

"메모리"는 사실 LangGraph의 **상태**임. 체크포인터에 저장된 thread의 상태를 직접 꺼내볼 수 있음.

In [ ]:
# 특정 thread의 현재 상태(누적된 messages) 확인
state = agent.get_state({"configurable": {"thread_id": "손님A"}})
print("손님A thread에 저장된 메시지 수:", len(state.values["messages"]))
for m in state.values["messages"]:
    role = type(m).__name__
    content = (m.content[:40] if m.content else "(도구 호출)")
    print(f"  [{role}] {content}")
print("\n→ 이 'messages를 담은 상태'가 바로 LangGraph가 관리하는 state. 고급(Part 12~)에서 직접 다룸.")

## ✏️ 미니 실습

**과제**: `thread_id="me"`로 다음 3턴을 이어서 대화하기.
1. "내 이름은 춘현이야"
2. "베이지 니트 추천해줘"
3. "내 이름 기억해?"  ← 1번을 기억하는지 확인

In [ ]:
# TODO: 직접 작성
# cfg_me = {"configurable": {"thread_id": "me"}}
# agent.invoke({"messages": [("user", "내 이름은 춘현이야")]}, config=cfg_me)
# ...
# print(agent.invoke({"messages": [("user", "내 이름 기억해?")]}, config=cfg_me)["messages"][-1].content)

<details><summary>👉 정답</summary>

```python
cfg_me = {"configurable": {"thread_id": "me"}}
agent.invoke({"messages": [("user", "내 이름은 춘현이야")]}, config=cfg_me)
agent.invoke({"messages": [("user", "베이지 니트 추천해줘")]}, config=cfg_me)
r = agent.invoke({"messages": [("user", "내 이름 기억해?")]}, config=cfg_me)
print(r["messages"][-1].content)   # "춘현님" 이라고 기억해야 함
```
</details>

## 정리

| 절 | 한 일 | 핵심 |
|---|---|---|
| 1 | 기억 없는 한계 | 두 호출이 독립적 |
| 2 | 체크포인터 | `InMemorySaver()` 장착 |
| 3 | thread_id | 같으면 맥락 유지 |
| 4 | thread 분리 | 다르면 새 대화 |
| 5 | 여러 사용자 | thread별 분리로 동시 응대 |
| 6 | 상태 | 메모리 = LangGraph state |

### 3줄 요약
1. `checkpointer=InMemorySaver()` + `config`의 `thread_id`로 메모리가 작동함.
2. 같은 thread_id면 맥락 유지, 다르면 분리 → 여러 사용자 동시 응대 가능.
3. 메모리는 곧 LangGraph의 상태이며, 고급에서 직접 다룸.

### ▶ 다음 (Part 11 · 중급 졸업)
도구·RAG·메모리를 모두 합친 **중급 프로젝트**(기억하는 상담/리서치 에이전트)를 완성하고, 그 한계를 마주하며 고급(LangGraph)으로 넘어감.